# Golden set

Construccion del conjunto de evaluacion que sirve como criterio de aceptacion (precision/recall >= 95%).

Flujo:
1. Carga y deduplicacion por imagen
2. Estratos de razon
3. Muestreo estratificado desde ambas clases del sistema legacy
4. Descarga de imagenes
5. Pre-etiquetado con el VLM (se corre aparte, ver scripts/run_vlm_annotation.py)
6. Revision y correccion humana
7. Congelar el golden set

In [1]:
import io
import hashlib
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import requests
from PIL import Image as PILImage

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
GOLDEN_DIR = DATA / "golden"
IMAGES_DIR = GOLDEN_DIR / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
pd.set_option("display.max_colwidth", None)

## 1. Carga y deduplicacion

In [ ]:
data = pd.read_csv(DATA / "anexo_1_dataset.csv")

counts = data.groupby("picture_url")["infraction_detected"].nunique()
inconsistent = counts[counts > 1].index

df = data[~data["picture_url"].isin(inconsistent)].drop_duplicates("picture_url").reset_index(drop=True)
print("crudo:", len(data), "| deduplicado:", len(df))
print(df["infraction_detected"].value_counts().to_dict())

## 2. Estratos de razon

In [ ]:
def reason_buckets(labels):
    s = str(labels).lower()
    out = set()
    if any(k in s for k in ["envio", "env\u00edo", "llega", "entrega", "inmediat", "pronta"]):
        out.add("delivery")
    if any(k in s for k in ["gratis", "oferta", "descuento", "promo", "black", "hot", "cuotas", "off", "sale", "mejor", "#1", "mas vendido"]):
        out.add("promo")
    if any(k in s for k in ["meli_label", "recomendado", "mercado lider", "official", "compra segura", "meli+", "destacado", "validac", "platinum", "gold", "amz"]):
        out.add("badge")
    return out

promo_kw = ["gratis", "oferta", "descuento", "% off", " off", "envio", "env\u00edo",
            "llega hoy", "promo", "black friday", "hot sale", "cuotas", "sin interes", "2x1", "liquidacion"]

pos = df[df["infraction_detected"]].copy()
pos["bk"] = pos["labels_detected"].apply(reason_buckets)

neg = df[~df["infraction_detected"]].copy()
neg["ocr"] = neg["ocr_text"].fillna("").astype(str).str.lower()
neg["promo_like"] = neg["ocr"].apply(lambda t: any(k in t for k in promo_kw))

print("cobertura buckets en positivos:")
for b in ["promo", "delivery", "badge"]:
    print(b, round(pos["bk"].apply(lambda s: b in s).mean(), 3))
print("aceptadas con ocr promocional:", int(neg["promo_like"].sum()))

## 3. Muestreo estratificado

In [ ]:
used = set()

def sample_stratum(frame, n, stratum):
    pool = frame[~frame["picture_url"].isin(used)]
    s = pool.sample(min(n, len(pool)), random_state=SEED).assign(source_stratum=stratum)
    used.update(s["picture_url"])
    return s

parts = [
    sample_stratum(pos[pos["bk"].apply(lambda s: "delivery" in s)], 80, "P-delivery"),
    sample_stratum(pos[pos["bk"].apply(lambda s: "badge" in s)], 120, "P-badge"),
    sample_stratum(pos[pos["bk"].apply(lambda s: "promo" in s)], 130, "P-promo"),
    sample_stratum(pos[pos["bk"].apply(lambda s: len(s) == 0)], 20, "P-sin_bucket"),
    sample_stratum(neg[~neg["promo_like"]], 200, "N-random"),
    sample_stratum(neg[neg["promo_like"]], 150, "N-dirigido"),
]

golden = pd.concat(parts).drop_duplicates("picture_url").reset_index(drop=True)
golden["legacy_label"] = golden["infraction_detected"]
for c in ["vlm_has_infraction", "vlm_reason", "vlm_evidence", "vlm_confidence",
          "human_label", "human_reason", "final_label", "notes"]:
    golden[c] = pd.NA

golden = golden[["picture_url", "legacy_label", "source_stratum", "labels_detected", "ocr_text",
                 "vlm_has_infraction", "vlm_reason", "vlm_evidence", "vlm_confidence",
                 "human_label", "human_reason", "final_label", "notes"]]

golden.to_parquet(GOLDEN_DIR / "golden_candidates.parquet", index=False)
print("total:", len(golden))
print(golden["source_stratum"].value_counts().to_dict())
print("legacy:", golden["legacy_label"].value_counts().to_dict())

## 4. Descarga de imagenes. Descarga concurrente con cache (no vuelve a bajar lo ya descargado) y rutas relativas a la raiz del proyecto.

In [ ]:
def url_id(u):
    return hashlib.md5(u.encode()).hexdigest()

sess = requests.Session()
sess.headers.update({"User-Agent": "Mozilla/5.0"})

def fetch(u):
    fn = url_id(u) + ".jpg"
    dest = IMAGES_DIR / fn
    rel = f"data/golden/images/{fn}"
    if dest.exists() and dest.stat().st_size > 0:
        return u, "cached", fn, rel
    for attempt in range(3):
        try:
            r = sess.get(u, timeout=10)
            if r.status_code == 200 and r.content:
                dest.write_bytes(r.content)
                return u, "ok", fn, rel
            return u, f"http_{r.status_code}", None, None
        except Exception as e:
            if attempt == 2:
                return u, f"err_{type(e).__name__}", None, None
            time.sleep(0.5)

golden = pd.read_parquet(GOLDEN_DIR / "golden_candidates.parquet")
rows = []
with ThreadPoolExecutor(max_workers=16) as ex:
    futs = [ex.submit(fetch, u) for u in golden["picture_url"]]
    for f in as_completed(futs):
        rows.append(f.result())

man = pd.DataFrame(rows, columns=["picture_url", "status", "filename", "local_path"])
man.to_parquet(GOLDEN_DIR / "images_manifest.parquet", index=False)
print(man["status"].value_counts().to_dict())
print("paths validos:", int(man["local_path"].notna().sum()), "/", len(man))

## 5. Pre-etiquetado con el VLM

Este paso es largo (~75 min en M3) y se corre por separado, fuera del notebook, para no consumir la sesion. Usa el VLM local Qwen2.5-VL-3B (anotador 1).

Desde la raiz del proyecto:

```
/opt/anaconda3/envs/computer-vision-poc/bin/python scripts/run_vlm_annotation.py
```

Lee `golden_candidates.parquet`, escribe `golden_annotated.parquet` con las columnas `vlm_*`. Es resumible: si se corta, vuelve a correrlo y continua donde quedo.

## 6. Revision y correccion (adjudicacion humana)

Agreement-based labeling. Solo se adjudican a mano los casos en **conflicto** (VLM discrepa de legacy) y los errores de parseo del VLM. Donde VLM y legacy **coinciden** se toma como verdad (silver), y se valida con un **spot-check** aleatorio de ese bucket. Asi la revision baja de ~450 a ~150 sin perder confiabilidad. Cada decision se guarda al instante en `golden_annotated.parquet`.

In [2]:
import ipywidgets as widgets
from IPython.display import display


class GoldenReviewer:
    """Agreement-based labeling, ajustado tras el spot-check. Se adjudican:
    conflictos VLM-vs-legacy, errores de parseo, y el bucket donde ambos
    coinciden en INFRACCION (~37% resultaron falsos positivos del sobre-marcado
    compartido). El bucket donde ambos coinciden en 'limpio' (0% de error en el
    spot-check) se toma como verdad y solo se valida con una muestra."""

    def __init__(self, spot_check=20):
        self.path = GOLDEN_DIR / "golden_annotated.parquet"
        self.g = pd.read_parquet(self.path)
        man = pd.read_parquet(GOLDEN_DIR / "images_manifest.parquet")
        self.rel = man.set_index("picture_url")["local_path"].to_dict()

        for col in ["vlm_has_infraction", "legacy_label", "human_label", "final_label"]:
            if col not in self.g.columns:
                self.g[col] = pd.NA
            self.g[col] = self.g[col].astype("boolean")

        m = self.g["final_label"].isna() & self.g["vlm_has_infraction"].notna()
        if m.any():
            self.g.loc[m, "final_label"] = self.g.loc[m, "vlm_has_infraction"]

        self.queue = self._build_queue(spot_check)
        self.i = 0
        self._build_ui()

    def _build_queue(self, spot_check):
        g = self.g
        vlm = g["vlm_has_infraction"]
        leg = g["legacy_label"]
        pending = g["human_label"].isna()

        conflict = (vlm != leg).fillna(False) & pending
        parse_error = vlm.isna() & pending
        agree_true = (vlm & leg).fillna(False) & pending
        priority = list(g[conflict | parse_error | agree_true].index)

        agree_false_pool = list(g[(~vlm & ~leg).fillna(False) & pending].index)
        n = min(spot_check, len(agree_false_pool))
        sample = list(pd.Series(agree_false_pool).sample(n, random_state=SEED)) if n else []
        self.spot_check_ids = set(sample)

        print(f"cola: {len(priority)} (conflictos + agree-TRUE + errores) + {len(sample)} spot-check limpio = {len(priority) + len(sample)}")
        return priority + sample

    def _img_bytes(self, picture_url):
        rel = self.rel.get(picture_url)
        if not rel:
            return b""
        try:
            im = PILImage.open(ROOT / rel).convert("RGB")
            im.thumbnail((400, 400))
            buf = io.BytesIO()
            im.save(buf, format="PNG")
            return buf.getvalue()
        except Exception:
            return b""

    def _build_ui(self):
        self.info = widgets.HTML()
        self.img = widgets.Image(format="png")
        b_inf = widgets.Button(description="Infraccion", button_style="danger")
        b_ok = widgets.Button(description="No infraccion", button_style="success")
        b_skip = widgets.Button(description="Saltar")
        b_inf.on_click(lambda _: self._set(True))
        b_ok.on_click(lambda _: self._set(False))
        b_skip.on_click(lambda _: self._advance())
        display(widgets.VBox([self.info, self.img, widgets.HBox([b_inf, b_ok, b_skip])]))
        self._show()

    def _show(self):
        if self.i >= len(self.queue):
            self.info.value = "<b>Cola de revision terminada.</b>"
            self.img.value = b""
            return
        idx = self.queue[self.i]
        r = self.g.loc[idx]
        tag = "spot-check" if idx in self.spot_check_ids else "conflicto/agree-TRUE"
        self.img.value = self._img_bytes(r["picture_url"])
        self.info.value = (
            f"<b>{self.i + 1}/{len(self.queue)}</b> &nbsp; [{tag}] &nbsp; estrato: {r['source_stratum']}<br>"
            f"legacy: <b>{r['legacy_label']}</b> &nbsp; vlm: <b>{r['vlm_has_infraction']}</b> ({r['vlm_confidence']})<br>"
            f"vlm_reason: {r['vlm_reason']}<br>"
            f"vlm_evidence: {r['vlm_evidence']}<br>"
            f"ocr: {str(r['ocr_text'])[:200]}"
        )

    def _set(self, value):
        idx = self.queue[self.i]
        self.g.at[idx, "human_label"] = value
        self.g.at[idx, "final_label"] = value
        self.g.to_parquet(self.path, index=False)
        self._advance()

    def _advance(self):
        self.i += 1
        self._show()


reviewer = GoldenReviewer(spot_check=20)

cola: 207 (conflictos + agree-TRUE + errores) + 20 spot-check limpio = 227


## 7. Congelar el golden set. Calcula la tasa de falsos positivos/negativos del sistema legacy contra la verdad humana y guarda el golden set final.

In [3]:
g = pd.read_parquet(GOLDEN_DIR / "golden_annotated.parquet")

P = g[g["source_stratum"].str.startswith("P")]
N = g[g["source_stratum"].str.startswith("N")]

legacy_fp = (P["final_label"] == False).mean()
legacy_fn = (N["final_label"] == True).mean()
agree_vlm_human = (g["vlm_has_infraction"].astype("boolean") == g["final_label"].astype("boolean")).mean()

print("tasa de falsos positivos del legacy (en estratos P):", round(float(legacy_fp), 3))
print("tasa de falsos negativos del legacy (en estratos N):", round(float(legacy_fn), 3))
print("acuerdo VLM vs humano:", round(float(agree_vlm_human), 3))
print("distribucion final_label:", g["final_label"].value_counts(dropna=False).to_dict())

g.to_parquet(GOLDEN_DIR / "golden_set.parquet", index=False)
print("golden set congelado en data/golden/golden_set.parquet")

tasa de falsos positivos del legacy (en estratos P): 0.446
tasa de falsos negativos del legacy (en estratos N): 0.0
acuerdo VLM vs humano: 0.815
distribucion final_label: {np.False_: 506, np.True_: 194}
golden set congelado en data/golden/golden_set.parquet
